# Taller: de Google Colab a GitHub, Render, FastAPI y Flutter

**Objetivo:** convertir grados Celsius a Fahrenheit con Python, publicar la función como API REST y consumirla desde Flutter.

Ejecuta las celdas en orden, presionando **Shift + Enter**. No necesitas activar GPU ni vincular Google Drive.

## Paso 1. Crear la función en Python
Usamos la fórmula: **°F = (°C × 9/5) + 32**.

In [ ]:
def celsius_a_fahrenheit(celsius):
    return round((celsius * 9 / 5) + 32, 2)

# Ejemplo que mostraremos durante el taller:
temperatura = 25
resultado = celsius_a_fahrenheit(temperatura)
print(f"{temperatura} °C = {resultado:.2f} °F")

## Paso 2. Comprobar otros valores
Si todas las pruebas pasan, ya sabemos que nuestra función básica funciona.

In [ ]:
assert celsius_a_fahrenheit(0) == 32.0
assert celsius_a_fahrenheit(25) == 77.0
assert celsius_a_fahrenheit(100) == 212.0
print("Correcto: las tres conversiones funcionan.")

## Paso 3. Exportar la función como archivos de una API FastAPI
Esta celda crea `conversor.py`, `main.py` y `requirements.txt` automáticamente. **El archivo principal que Render ejecutará será `main.py`.**

In [ ]:
from pathlib import Path

carpeta = Path("api_temperatura")
carpeta.mkdir(exist_ok=True)

conversor_codigo = '"""Lógica del taller: convertir grados Celsius a Fahrenheit."""\n\n\ndef celsius_a_fahrenheit(celsius: float) -> float:\n    """Convierte °C a °F y redondea el resultado a dos decimales."""\n    return round((celsius * 9 / 5) + 32, 2)\n\n\nif __name__ == "__main__":\n    temperatura = 25\n    print(f"{temperatura} °C = {celsius_a_fahrenheit(temperatura):.2f} °F")\n'
main_codigo = '"""API REST de demostración para Google Colab -> GitHub -> Render -> Flutter."""\n\nfrom fastapi import FastAPI\nfrom fastapi.middleware.cors import CORSMiddleware\nfrom pydantic import BaseModel, Field\n\nfrom conversor import celsius_a_fahrenheit\n\napp = FastAPI(\n    title="API de Temperaturas - Taller Python a Flutter",\n    description="Convierte grados Celsius a Fahrenheit para la práctica del taller.",\n    version="1.0.0",\n)\n\n# Para la demostración en Flutter Web. En una API real, limitar orígenes.\napp.add_middleware(\n    CORSMiddleware,\n    allow_origins=["*"],\n    allow_credentials=False,\n    allow_methods=["GET", "POST"],\n    allow_headers=["Content-Type"],\n)\n\n\nclass TemperaturaEntrada(BaseModel):\n    celsius: float = Field(..., description="Temperatura expresada en grados Celsius")\n\n\n@app.get("/", tags=["Estado"])\ndef inicio():\n    return {"mensaje": "API activa", "documentacion": "/docs"}\n\n\n@app.post("/convertir", tags=["Conversión"])\ndef convertir(datos: TemperaturaEntrada):\n    return {\n        "celsius": datos.celsius,\n        "fahrenheit": celsius_a_fahrenheit(datos.celsius),\n    }\n'
requisitos_codigo = 'fastapi>=0.115,<1.0\nuvicorn>=0.30,<1.0\n'

(carpeta / "conversor.py").write_text(conversor_codigo, encoding="utf-8")
(carpeta / "main.py").write_text(main_codigo, encoding="utf-8")
(carpeta / "requirements.txt").write_text(requisitos_codigo, encoding="utf-8")

print("Archivos creados:")
for archivo in sorted(carpeta.iterdir()):
    print(" -", archivo.name)

## Paso 4. Probar FastAPI sin abrir un servidor en Colab
Instalamos las dependencias y usamos el cliente de pruebas de FastAPI. **No necesitas crear un túnel web de Colab** porque el servicio público estará en Render.

In [ ]:
%pip -q install -r api_temperatura/requirements.txt httpx

In [ ]:
import sys
from pathlib import Path
from fastapi.testclient import TestClient

sys.path.insert(0, str(Path("api_temperatura").resolve()))
from main import app

cliente = TestClient(app)
respuesta_inicio = cliente.get("/")
respuesta_conversion = cliente.post("/convertir", json={"celsius": 25})

assert respuesta_inicio.status_code == 200
assert respuesta_conversion.status_code == 200
assert respuesta_conversion.json()["fahrenheit"] == 77.0
print("GET /:", respuesta_inicio.json())
print("POST /convertir:", respuesta_conversion.json())
print("API probada correctamente")

## Paso 5. Descargar el ZIP para GitHub
Se descarga un ZIP con **tres archivos en su raíz**. En GitHub tendrás que descomprimir el ZIP y subir esos tres archivos, no subir el ZIP como un solo archivo.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED

ruta_zip = Path("api_temperatura_para_github.zip")
with ZipFile(ruta_zip, "w", ZIP_DEFLATED) as paquete:
    for nombre in ("conversor.py", "main.py", "requirements.txt"):
        paquete.write(carpeta / nombre, arcname=nombre)

print(f"ZIP generado: {ruta_zip.resolve()}")
from google.colab import files
files.download(str(ruta_zip))